In [ ]:
# Notebook 07: Optuna Hyperparameter Tuning + ConvNeXt-Nano Manual Fix Retrain

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from sklearn.metrics import (
    f1_score, recall_score, roc_auc_score,
    roc_curve, confusion_matrix
)
from sklearn.calibration import calibration_curve

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"timm    : {timm.__version__}")
print(f"optuna  : {optuna.__version__}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

## Paths and control flags

In [ ]:
NB02 = Path("/kaggle/input/notebooks/mfjmrizvi/02-mil-patch-extraction")
NB03 = Path("/kaggle/input/notebooks/mfjmrizvi/03-efficientnet-b0")
NB05 = Path("/kaggle/input/notebooks/mfjmrizvi/05-swint")
OUT  = Path("/kaggle/working")

X_TRAIN_PATH       = NB02 / "X_train_patches.npy"
Y_TRAIN_PATH       = NB02 / "y_train_labels.npy"
BAG_IDS_TRAIN_PATH = NB02 / "bag_ids_train.npy"
FOLD_IDS_PATH        = NB02 / "fold_ids.npy"  
CLASS_WEIGHTS_PATH   = NB02 / "class_weights.json"

EFFNET_S1_WEIGHTS = NB03 / "efficientnet_b0_stage1.pth"
SWINT_S1_WEIGHTS  = NB05 / "swin_t_stage1.pth"

# ── Control flags ──
STAGE1_FOLD = 0   # match NB03/04's fixed-fold convention
N_TRIALS_EFFNET   = 25   # reduce to 15 if GPU budget tight (Mid-Term 3.9.3)
N_TRIALS_SWINT    = 25
RUN_CONVNEXT_OPTUNA = True
N_TRIALS_CONVNEXT = 25

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

import sys
sys.path.append('/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config')
from abmil_common import (
    build_backbone, PatchDataset, PatchClassifier,
    AttentionPool, BagClassifier, CachedBagDataset, collate_cached,
    extract_features, compute_all_metrics, get_normalisation_tensors,
)
_mean_gpu, _std_gpu = get_normalisation_tensors(DEVICE)

In [ ]:
import inspect
import abmil_common
print(inspect.getsource(abmil_common.AttentionPool))

In [ ]:
# Cell — Verify abmil_common.py source in current session
import inspect
from abmil_common import AttentionPool

source = inspect.getsource(AttentionPool)
print(source)

# Check specifically for the fix
if "nn.Identity()" in source:
    print("\n✓ Fixed version confirmed — nn.Identity() used for dropout<=0")
else:
    print("\n✗ STALE VERSION — fix not present. Re-attach the correct dataset version before proceeding.")

In [ ]:
import os
path = '/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config'
print(os.listdir(path))
print(os.path.getmtime(f'{path}/abmil_common.py'))

Cell 3 : Load NB02 data (identical to NB03/04/05)

In [ ]:
# Cell 3 : Load NB02 data (derive train/val from fold_ids, not saved index files)
X_train_all  = np.load(X_TRAIN_PATH)
y_train_all  = np.load(Y_TRAIN_PATH)
bag_ids_all  = np.load(BAG_IDS_TRAIN_PATH)
fold_ids     = np.load(FOLD_IDS_PATH)

with open(CLASS_WEIGHTS_PATH) as f:
    raw_cw = json.load(f)
class_weight_dict = {int(k): float(v) for k, v in raw_cw.items()}

all_bags = np.unique(bag_ids_all)
train_bag_indices = all_bags[fold_ids[all_bags] != STAGE1_FOLD]
val_bag_indices   = all_bags[fold_ids[all_bags] == STAGE1_FOLD]

train_mask = np.isin(bag_ids_all, train_bag_indices)
val_mask   = np.isin(bag_ids_all, val_bag_indices)

X_tr, y_tr, bag_tr    = X_train_all[train_mask], y_train_all[train_mask], bag_ids_all[train_mask]
X_val, y_val, bag_val = X_train_all[val_mask],   y_train_all[val_mask],   bag_ids_all[val_mask]
# X_test_all / y_test_all / bag_ids_test removed entirely

print(f"Train patches: {X_tr.shape}  Val patches: {X_val.shape}")
print(f"Class weights: {class_weight_dict}")

Part A : ConvNeXt-Nano manual hyperparameter fix retrain (ISS-001)
Cell 5 : Stage 1 retrain with corrected hyperparameters

In [ ]:
MODEL_NAME  = "convnext_nano"
ATTN_DIM    = 128
PATCH_SIZE  = 224

BS_STAGE1   = 128        # unchanged — held constant for cross-comparison
LR_STAGE1   = 1e-4       # was 1e-3
EPOCHS_S1   = 25         # was 15
PATIENCE_S1 = 7          # was 5
LR_S2_HEAD  = 5e-5       # was 1e-4 — fixed, not tuned (manual fix only)

S1_WEIGHTS_V2   = OUT / "convnext_nano_v2_stage1.pth"
S2_WEIGHTS_V2   = OUT / "convnext_nano_v2_stage2.pth"
RESULTS_JSON_V2 = OUT / "convnext_nano_v2_results.json"

backbone = build_backbone(MODEL_NAME, pretrained=True).to(DEVICE)
with torch.no_grad():
    _dummy = torch.zeros(2, 3, PATCH_SIZE, PATCH_SIZE, device=DEVICE)
    FEAT_DIM = backbone(_dummy).shape[1]
del _dummy
print(f"FEAT_DIM confirmed: {FEAT_DIM}")

patch_model = PatchClassifier(backbone, FEAT_DIM).to(DEVICE)
pos_weight_val = class_weight_dict[1] / class_weight_dict[0]
criterion_s1 = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_val, device=DEVICE))
optimiser_s1 = optim.Adam(patch_model.parameters(), lr=LR_STAGE1)
scheduler_s1 = optim.lr_scheduler.ReduceLROnPlateau(optimiser_s1, mode='min', factor=0.5, patience=3)

train_patch_dl = DataLoader(PatchDataset(X_tr, y_tr), batch_size=BS_STAGE1, shuffle=True, num_workers=2, pin_memory=True)
val_patch_dl   = DataLoader(PatchDataset(X_val, y_val), batch_size=BS_STAGE1, shuffle=False, num_workers=2, pin_memory=True)

scaler = torch.amp.GradScaler('cuda')
s1_history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_f1": []}
best_val_loss_s1, patience_counter_s1 = float("inf"), 0
t0 = time.time()

for epoch in range(1, EPOCHS_S1 + 1):
    patch_model.train()
    running_loss = 0.0
    for patches, labels in train_patch_dl:
        patches, labels = patches.to(DEVICE), labels.to(DEVICE)
        patches = patches.repeat(1, 3, 1, 1)
        patches = (patches - _mean_gpu) / _std_gpu
        optimiser_s1.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = patch_model(patches)
            loss = criterion_s1(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimiser_s1)
        scaler.update()
        running_loss += loss.item() * len(labels)
    train_loss = running_loss / len(train_patch_dl.dataset)

    patch_model.eval()
    val_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for patches, labels in val_patch_dl:
            patches, labels = patches.to(DEVICE), labels.to(DEVICE)
            patches = patches.repeat(1, 3, 1, 1)
            patches = (patches - _mean_gpu) / _std_gpu
            with torch.amp.autocast('cuda'):
                logits = patch_model(patches)
                loss_val = criterion_s1(logits, labels)
            val_loss += loss_val.item() * len(labels)
            all_probs.extend(torch.sigmoid(logits).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    val_loss /= len(val_patch_dl.dataset)
    all_probs, all_labels = np.array(all_probs), np.array(all_labels)
    val_preds = (all_probs >= 0.5).astype(int)
    val_auc = roc_auc_score(all_labels, all_probs)
    val_f1  = f1_score(all_labels, val_preds, zero_division=0)

    s1_history["train_loss"].append(train_loss)
    s1_history["val_loss"].append(val_loss)
    s1_history["val_auc"].append(val_auc)
    s1_history["val_f1"].append(val_f1)
    scheduler_s1.step(val_loss)

    print(f"Ep {epoch:02d}/{EPOCHS_S1}  train={train_loss:.4f}  val={val_loss:.4f}  auc={val_auc:.4f}  f1={val_f1:.4f}")

    if val_loss < best_val_loss_s1:
        best_val_loss_s1, patience_counter_s1 = val_loss, 0
        torch.save(patch_model.state_dict(), S1_WEIGHTS_V2)
        print("  ✓ Saved best Stage 1 weights (v2)")
    else:
        patience_counter_s1 += 1
        if patience_counter_s1 >= PATIENCE_S1:
            print(f"  Early stopping at epoch {epoch}")
            break

print(f"\nStage 1 (v2) complete — {(time.time()-t0)/60:.1f} min")

Cell 6 : Stage 1 (v2) patch-level diagnostics — compare against ISS-001 baseline (patch AUC 0.7250)

In [ ]:
patch_model.load_state_dict(torch.load(S1_WEIGHTS_V2, map_location=DEVICE))
patch_model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for patches, labels in val_patch_dl:
        patches = patches.to(DEVICE)
        patches = patches.repeat(1, 3, 1, 1)
        patches = (patches - _mean_gpu) / _std_gpu
        with torch.amp.autocast('cuda'):
            all_probs.extend(torch.sigmoid(patch_model(patches)).cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs, all_labels = np.array(all_probs), np.array(all_labels)
preds = (all_probs >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()

s1_metrics_v2 = {
    "patch_auc": float(roc_auc_score(all_labels, all_probs)),
    "patch_f1": float(f1_score(all_labels, preds, zero_division=0)),
    "patch_sensitivity": float(recall_score(all_labels, preds, zero_division=0)),
    "patch_specificity": float(tn / (tn + fp)),
}
print("── ConvNeXt-Nano Stage 1 (v2) Patch-Level Val Metrics ──")
for k, v in s1_metrics_v2.items():
    print(f"  {k:25s}: {v:.4f}")
print(f"\n  Baseline (NB04) patch_auc was 0.7250")

Cell 7 : Feature extraction (cached to disk this time, for later reuse if needed)

In [ ]:
# feature_extractor = patch_model.backbone
# feature_extractor.eval()
# for p in feature_extractor.parameters():
#     p.requires_grad_(False)

# print("Extracting ConvNeXt-Nano (v2) features...")
# feats_tr_cnx   = extract_features(X_tr, feature_extractor, _mean_gpu, _std_gpu, DEVICE)
# feats_val_cnx  = extract_features(X_val, feature_extractor, _mean_gpu, _std_gpu, DEVICE)
# feats_test_cnx = extract_features(X_test_all, feature_extractor, _mean_gpu, _std_gpu, DEVICE)
# print(f"  train : {feats_tr_cnx.shape}  val : {feats_val_cnx.shape}  test : {feats_test_cnx.shape}")

# np.save(OUT / "convnext_nano_v2_feats_tr.npy", feats_tr_cnx)
# np.save(OUT / "convnext_nano_v2_feats_val.npy", feats_val_cnx)
# np.save(OUT / "convnext_nano_v2_feats_test.npy", feats_test_cnx)

# Part A, Cell 7: feature extraction — train + validation ONLY

feature_extractor = patch_model.backbone
feature_extractor.eval()

for p in feature_extractor.parameters():
    p.requires_grad_(False)

print("Extracting ConvNeXt-Nano (v2) features...")

feats_tr_cnx = extract_features(
    X_tr,
    feature_extractor,
    _mean_gpu,
    _std_gpu,
    DEVICE
)

feats_val_cnx = extract_features(
    X_val,
    feature_extractor,
    _mean_gpu,
    _std_gpu,
    DEVICE
)

print(
    f"  train : {feats_tr_cnx.shape}"
    f"  val   : {feats_val_cnx.shape}"
)

np.save(
    OUT / "convnext_nano_v2_feats_tr.npy",
    feats_tr_cnx
)

np.save(
    OUT / "convnext_nano_v2_feats_val.npy",
    feats_val_cnx
)

Cell 8 : Stage 2 training with fixed manual-fix hyperparameters (no Optuna)

In [ ]:
bag_model = BagClassifier(FEAT_DIM, ATTN_DIM, dropout=0.0).to(DEVICE)
criterion_s2 = nn.BCEWithLogitsLoss()
optimiser_s2 = optim.Adam(bag_model.parameters(), lr=LR_S2_HEAD)
scheduler_s2 = optim.lr_scheduler.ReduceLROnPlateau(optimiser_s2, mode='min', factor=0.5, patience=3)

EPOCHS_S2, PATIENCE_S2 = 30, 7

train_bag_ds = CachedBagDataset(feats_tr_cnx, y_tr, bag_tr, train_bag_indices)
val_bag_ds   = CachedBagDataset(feats_val_cnx, y_val, bag_val, val_bag_indices)
train_bag_dl = DataLoader(train_bag_ds, batch_size=1, shuffle=True, collate_fn=collate_cached)
val_bag_dl   = DataLoader(val_bag_ds, batch_size=1, shuffle=False, collate_fn=collate_cached)

s2_history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_f1": []}
best_val_loss_s2, patience_counter_s2 = float("inf"), 0
t0 = time.time()

for epoch in range(1, EPOCHS_S2 + 1):
    bag_model.train()
    running_loss = 0.0
    for h_list, labels in train_bag_dl:
        h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
        optimiser_s2.zero_grad()
        logit, _ = bag_model(h)
        loss = criterion_s2(logit, label)
        loss.backward()
        optimiser_s2.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_bag_ds)

    bag_model.eval()
    val_loss, val_probs, val_true = 0.0, [], []
    with torch.no_grad():
        for h_list, labels in val_bag_dl:
            h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
            logit, _ = bag_model(h)
            val_loss += criterion_s2(logit, label).item()
            val_probs.append(torch.sigmoid(logit).item())
            val_true.append(label.item())
    val_loss /= len(val_bag_ds)
    val_probs, val_true = np.array(val_probs), np.array(val_true)
    val_preds = (val_probs >= 0.5).astype(int)
    val_auc = roc_auc_score(val_true, val_probs)
    val_f1  = f1_score(val_true, val_preds, zero_division=0)

    s2_history["train_loss"].append(train_loss)
    s2_history["val_loss"].append(val_loss)
    s2_history["val_auc"].append(val_auc)
    s2_history["val_f1"].append(val_f1)
    scheduler_s2.step(val_loss)

    print(f"Ep {epoch:02d}/{EPOCHS_S2}  train={train_loss:.4f}  val={val_loss:.4f}  auc={val_auc:.4f}  f1={val_f1:.4f}")

    if val_loss < best_val_loss_s2:
        best_val_loss_s2, patience_counter_s2 = val_loss, 0
        torch.save(bag_model.state_dict(), S2_WEIGHTS_V2)
        print("  ✓ Saved best Stage 2 (v2) weights")
    else:
        patience_counter_s2 += 1
        if patience_counter_s2 >= PATIENCE_S2:
            print(f"  Early stopping at epoch {epoch}")
            break

print(f"\nStage 2 (v2) complete — {(time.time()-t0)/60:.1f} min")

Cell 9 : ConvNeXt-Nano (v2) val/test evaluation, threshold calibration, save results

In [ ]:
# bag_model.load_state_dict(torch.load(S2_WEIGHTS_V2, map_location=DEVICE))
# bag_model.eval()

# val_bag_probs, val_bag_true = [], []
# with torch.no_grad():
#     for h_list, labels in val_bag_dl:
#         h = h_list[0].to(DEVICE)
#         logit, _ = bag_model(h)
#         val_bag_probs.append(torch.sigmoid(logit).item())
#         val_bag_true.append(labels[0].item())
# val_bag_probs, val_bag_true = np.array(val_bag_probs), np.array(val_bag_true)

# fpr_c, tpr_c, thresholds_c = roc_curve(val_bag_true, val_bag_probs)
# idx = np.argmax(tpr_c >= 0.90)
# optimal_threshold = float(thresholds_c[idx])
# print(f"Calibrated threshold: {optimal_threshold:.4f}  sensitivity@cal: {tpr_c[idx]:.4f}  FPR@cal: {fpr_c[idx]:.4f}")

# val_metrics_default = compute_all_metrics(val_bag_true, val_bag_probs, 0.5, "Val default (0.5)")
# val_metrics_cal = compute_all_metrics(val_bag_true, val_bag_probs, optimal_threshold, f"Val calibrated ({optimal_threshold:.3f})")

# test_bag_list = np.unique(bag_ids_test)
# test_bag_ds = CachedBagDataset(feats_test_cnx, y_test_all, bag_ids_test, test_bag_list)
# test_bag_dl = DataLoader(test_bag_ds, batch_size=1, shuffle=False, collate_fn=collate_cached)

# test_bag_probs, test_bag_true = [], []
# with torch.no_grad():
#     for h_list, labels in test_bag_dl:
#         h = h_list[0].to(DEVICE)
#         logit, _ = bag_model(h)
#         test_bag_probs.append(torch.sigmoid(logit).item())
#         test_bag_true.append(labels[0].item())
# test_bag_probs, test_bag_true = np.array(test_bag_probs), np.array(test_bag_true)

# test_metrics = compute_all_metrics(test_bag_true, test_bag_probs, optimal_threshold, f"Test calibrated ({optimal_threshold:.3f})")

# results_v2 = {
#     "model": "convnext_nano_v2_manual_fix",
#     "feat_dim": FEAT_DIM,
#     "attn_dim": ATTN_DIM,
#     "stage1_metrics": s1_metrics_v2,
#     "val_metrics_default": val_metrics_default,
#     "val_metrics_cal": val_metrics_cal,
#     "test_metrics": test_metrics,
#     "optimal_threshold": optimal_threshold,
#     "hyperparameters": {
#         "lr_stage1": LR_STAGE1, "lr_s2_head": LR_S2_HEAD,
#         "epochs_s1": EPOCHS_S1, "epochs_s2": EPOCHS_S2,
#         "bs_stage1": BS_STAGE1, "attn_dim": ATTN_DIM,
#         "patience_s1": PATIENCE_S1, "patience_s2": PATIENCE_S2,
#     }
# }
# with open(RESULTS_JSON_V2, "w") as f:
#     json.dump(results_v2, f, indent=2)
# print("Saved:", RESULTS_JSON_V2)

# print("\n── Comparison: ConvNeXt-Nano baseline (NB04) vs manual fix (v2) ──")
# print(f"{'Metric':<20}{'Baseline (NB04)':>18}{'Manual fix (v2)':>18}")
# baseline_test = {"auc": 0.9748, "f1": 0.9141, "sensitivity": 0.9048, "specificity": 0.9524, "fpr": 0.0476}
# for k in ["auc", "f1", "sensitivity", "specificity", "fpr"]:
#     print(f"{k:<20}{baseline_test[k]:>18.4f}{test_metrics[k]:>18.4f}")



In [ ]:
# Part A, Cell 9 — Validation-only evaluation
# IMPORTANT: Test set is intentionally NOT loaded or evaluated here.

# Load the best Stage 2 weights
bag_model.load_state_dict(
    torch.load(S2_WEIGHTS_V2, map_location=DEVICE)
)
bag_model.eval()

# ---------------------------------------------------------
# 1. Validation inference
# ---------------------------------------------------------
val_bag_probs = []
val_bag_true = []

with torch.no_grad():
    for h_list, labels in val_bag_dl:
        h = h_list[0].to(DEVICE)
        logit, _ = bag_model(h)

        val_bag_probs.append(
            torch.sigmoid(logit).item()
        )
        val_bag_true.append(
            labels[0].item()
        )

val_bag_probs = np.array(val_bag_probs)
val_bag_true = np.array(val_bag_true)

print(f"Validation bags evaluated: {len(val_bag_true)}")
print(f"Validation malignant:      {np.sum(val_bag_true == 1)}")
print(f"Validation benign:         {np.sum(val_bag_true == 0)}")


# ---------------------------------------------------------
# 2. Determine operating threshold using VALIDATION ONLY
# ---------------------------------------------------------
fpr_c, tpr_c, thresholds_c = roc_curve(
    val_bag_true,
    val_bag_probs
)

# Select first threshold achieving sensitivity >= 0.90
valid_idx = np.where(tpr_c >= 0.90)[0]

if len(valid_idx) == 0:
    # Fallback if 90% sensitivity is not achieved
    idx = np.argmax(tpr_c)
else:
    idx = valid_idx[0]

optimal_threshold = float(thresholds_c[idx])

print(
    f"\nCalibrated threshold: {optimal_threshold:.4f}"
)
print(
    f"Sensitivity @ threshold: {tpr_c[idx]:.4f}"
)
print(
    f"FPR @ threshold:         {fpr_c[idx]:.4f}"
)


# ---------------------------------------------------------
# 3. Validation metrics
# ---------------------------------------------------------
val_metrics_default = compute_all_metrics(
    val_bag_true,
    val_bag_probs,
    0.5,
    "Val default (0.5)"
)

val_metrics_cal = compute_all_metrics(
    val_bag_true,
    val_bag_probs,
    optimal_threshold,
    f"Val calibrated ({optimal_threshold:.3f})"
)


# ---------------------------------------------------------
# 4. Save validation-only results
# ---------------------------------------------------------
results_v2 = {
    "model": "convnext_nano_v2_manual_fix",

    "feat_dim": FEAT_DIM,
    "attn_dim": ATTN_DIM,

    "stage1_metrics": s1_metrics_v2,

    "val_metrics_default": val_metrics_default,
    "val_metrics_cal": val_metrics_cal,

    "optimal_threshold": optimal_threshold,

    "hyperparameters": {
        "lr_stage1": LR_STAGE1,
        "lr_s2_head": LR_S2_HEAD,
        "epochs_s1": EPOCHS_S1,
        "epochs_s2": EPOCHS_S2,
        "bs_stage1": BS_STAGE1,
        "attn_dim": ATTN_DIM,
        "patience_s1": PATIENCE_S1,
        "patience_s2": PATIENCE_S2,
    }
}

with open(RESULTS_JSON_V2, "w") as f:
    json.dump(results_v2, f, indent=2)

print("\n✓ Validation evaluation complete")
print("✓ Saved:", RESULTS_JSON_V2)
print("✓ Test set was NOT evaluated")

Part B : Optuna tuning: EfficientNet-B0 and Swin-T (Stage 2 attention head only)
Cell 10 : Rebuild frozen backbones and cache features for both models

In [ ]:
def rebuild_and_cache(model_name, s1_weights_path, tag):
    backbone = build_backbone(model_name, pretrained=True).to(DEVICE)
    with torch.no_grad():
        _dummy = torch.zeros(2, 3, PATCH_SIZE, PATCH_SIZE, device=DEVICE)
        feat_dim = backbone(_dummy).shape[1]
    del _dummy

    patch_model = PatchClassifier(backbone, feat_dim).to(DEVICE)
    patch_model.load_state_dict(torch.load(s1_weights_path, map_location=DEVICE))
    feature_extractor = patch_model.backbone
    feature_extractor.eval()
    for p in feature_extractor.parameters():
        p.requires_grad_(False)

    feats_tr  = extract_features(X_tr,  feature_extractor, _mean_gpu, _std_gpu, DEVICE)
    feats_val = extract_features(X_val, feature_extractor, _mean_gpu, _std_gpu, DEVICE)

    np.save(OUT / f"{tag}_feats_tr.npy", feats_tr)
    np.save(OUT / f"{tag}_feats_val.npy", feats_val)

    print(f"{tag}: FEAT_DIM={feat_dim}  train={feats_tr.shape}  val={feats_val.shape}")
    return feats_tr, feats_val, feat_dim   # no feats_test returned

feats_tr_eff, feats_val_eff, FEAT_DIM_EFF   = rebuild_and_cache("efficientnet_b0", EFFNET_S1_WEIGHTS, "effnet_b0")
feats_tr_swin, feats_val_swin, FEAT_DIM_SWIN = rebuild_and_cache("swin_tiny_patch4_window7_224", SWINT_S1_WEIGHTS, "swin_t")

Cell 11 : Generic Optuna objective for Stage 2 attention head

In [ ]:
def make_objective(feat_dim, feats_tr, y_tr_, bag_tr_, train_bag_idx,
                    feats_val, y_val_, bag_val_, val_bag_idx, max_epochs=20, patience=5):

    def objective(trial):
        attn_dim   = trial.suggest_categorical("attn_dim", [64, 128, 256])
        lr         = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
        dropout    = trial.suggest_float("dropout", 0.0, 0.5)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)

        model = BagClassifier(feat_dim, attn_dim, dropout).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss()
        optimiser = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        train_ds = CachedBagDataset(feats_tr, y_tr_, bag_tr_, train_bag_idx)
        val_ds   = CachedBagDataset(feats_val, y_val_, bag_val_, val_bag_idx)
        train_dl = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_cached)
        val_dl   = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_cached)

        best_val_loss, patience_ctr = float("inf"), 0

        for epoch in range(1, max_epochs + 1):
            model.train()
            for h_list, labels in train_dl:
                h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
                optimiser.zero_grad()
                logit, _ = model(h)
                loss = criterion(logit, label)
                loss.backward()
                optimiser.step()

            model.eval()
            val_loss, val_probs, val_true = 0.0, [], []
            with torch.no_grad():
                for h_list, labels in val_dl:
                    h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
                    logit, _ = model(h)
                    val_loss += criterion(logit, label).item()
                    val_probs.append(torch.sigmoid(logit).item())
                    val_true.append(label.item())
            val_loss /= len(val_ds)

            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            if val_loss < best_val_loss:
                best_val_loss, patience_ctr = val_loss, 0
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    break

        return best_val_loss

    return objective

Cell 12 : Run Optuna study: EfficientNet-B0

In [ ]:
sampler_eff = TPESampler(seed=SEED)
pruner_eff  = MedianPruner(n_warmup_steps=5)
study_eff = optuna.create_study(direction="minimize", sampler=sampler_eff, pruner=pruner_eff)

objective_eff = make_objective(
    FEAT_DIM_EFF, feats_tr_eff, y_tr, bag_tr, train_bag_indices,
    feats_val_eff, y_val, bag_val, val_bag_indices
)

t0 = time.time()
study_eff.optimize(objective_eff, n_trials=N_TRIALS_EFFNET)
print(f"EfficientNet-B0 Optuna search complete — {(time.time()-t0)/60:.1f} min")
print("Best params:", study_eff.best_params)
print("Best val_loss:", study_eff.best_value)

with open(OUT / "effnet_b0_optuna_study.json", "w") as f:
    json.dump({"best_params": study_eff.best_params, "best_value": study_eff.best_value,
               "n_trials": len(study_eff.trials)}, f, indent=2)

Cell 13: Run Optuna study: Swin-T

In [ ]:
sampler_swin = TPESampler(seed=SEED)
pruner_swin  = MedianPruner(n_warmup_steps=5)
study_swin = optuna.create_study(direction="minimize", sampler=sampler_swin, pruner=pruner_swin)

objective_swin = make_objective(
    FEAT_DIM_SWIN, feats_tr_swin, y_tr, bag_tr, train_bag_indices,
    feats_val_swin, y_val, bag_val, val_bag_indices
)

t0 = time.time()
study_swin.optimize(objective_swin, n_trials=N_TRIALS_SWINT)
print(f"Swin-T Optuna search complete — {(time.time()-t0)/60:.1f} min")
print("Best params:", study_swin.best_params)
print("Best val_loss:", study_swin.best_value)

with open(OUT / "swin_t_optuna_study.json", "w") as f:
    json.dump({"best_params": study_swin.best_params, "best_value": study_swin.best_value,
               "n_trials": len(study_swin.trials)}, f, indent=2)

Cell 14 : Retrain final Stage 2 models with best hyperparameters, evaluate on val + test

In [ ]:
# def train_final_and_evaluate(feat_dim, best_params, feats_tr_, feats_val_,
#                               tag, model_name, epochs=30, patience=7):
#     attn_dim = best_params["attn_dim"]
#     lr = best_params["lr"]
#     dropout = best_params["dropout"]
#     weight_decay = best_params["weight_decay"]

#     model = BagClassifier(feat_dim, attn_dim, dropout).to(DEVICE)
#     criterion = nn.BCEWithLogitsLoss()
#     optimiser = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
#     scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, patience=3)

#     train_ds = CachedBagDataset(feats_tr_, y_tr, bag_tr, train_bag_indices)
#     val_ds   = CachedBagDataset(feats_val_, y_val, bag_val, val_bag_indices)
#     train_dl = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_cached)
#     val_dl   = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_cached)

#     weights_path = OUT / f"{tag}_optuna_stage2.pth"
#     best_val_loss, patience_ctr = float("inf"), 0

#     for epoch in range(1, epochs + 1):
#         model.train()
#         running_loss = 0.0
#         for h_list, labels in train_dl:
#             h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
#             optimiser.zero_grad()
#             logit, _ = model(h)
#             loss = criterion(logit, label)
#             loss.backward()
#             optimiser.step()
#             running_loss += loss.item()
#         train_loss = running_loss / len(train_ds)

#         model.eval()
#         val_loss, val_probs, val_true = 0.0, [], []
#         with torch.no_grad():
#             for h_list, labels in val_dl:
#                 h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
#                 logit, _ = model(h)
#                 val_loss += criterion(logit, label).item()
#                 val_probs.append(torch.sigmoid(logit).item())
#                 val_true.append(label.item())
#         val_loss /= len(val_ds)
#         val_probs, val_true = np.array(val_probs), np.array(val_true)
#         val_auc = roc_auc_score(val_true, val_probs)
#         scheduler.step(val_loss)

#         print(f"[{tag}] Ep {epoch:02d}/{epochs}  train={train_loss:.4f}  val={val_loss:.4f}  auc={val_auc:.4f}")

#         if val_loss < best_val_loss:
#             best_val_loss, patience_ctr = val_loss, 0
#             torch.save(model.state_dict(), weights_path)
#         else:
#             patience_ctr += 1
#             if patience_ctr >= patience:
#                 print(f"  Early stopping at epoch {epoch}")
#                 break

#     model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
#     model.eval()

#     val_bag_probs, val_bag_true = [], []
#     with torch.no_grad():
#         for h_list, labels in val_dl:
#             h = h_list[0].to(DEVICE)
#             logit, _ = model(h)
#             val_bag_probs.append(torch.sigmoid(logit).item())
#             val_bag_true.append(labels[0].item())
#     val_bag_probs, val_bag_true = np.array(val_bag_probs), np.array(val_bag_true)

#     fpr_c, tpr_c, thresholds_c = roc_curve(val_bag_true, val_bag_probs)
#     idx = np.argmax(tpr_c >= 0.90)
#     optimal_threshold = float(thresholds_c[idx])

#     val_metrics_cal = compute_all_metrics(val_bag_true, val_bag_probs, optimal_threshold, f"[{tag}] Val calibrated")

#     test_bag_list = np.unique(bag_ids_test)
#     test_ds = CachedBagDataset(feats_test_, y_test_all, bag_ids_test, test_bag_list)
#     test_dl = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_cached)
#     test_bag_probs, test_bag_true = [], []
#     with torch.no_grad():
#         for h_list, labels in test_dl:
#             h = h_list[0].to(DEVICE)
#             logit, _ = model(h)
#             test_bag_probs.append(torch.sigmoid(logit).item())
#             test_bag_true.append(labels[0].item())
#     test_bag_probs, test_bag_true = np.array(test_bag_probs), np.array(test_bag_true)
#     test_metrics = compute_all_metrics(test_bag_true, test_bag_probs, optimal_threshold, f"[{tag}] Test calibrated")

#     results = {
#         "model": model_name,
#         "feat_dim": feat_dim,
#         "optuna_best_params": best_params,
#         "val_metrics_cal": val_metrics_cal,
#         "test_metrics": test_metrics,
#         "optimal_threshold": optimal_threshold,
#     }
#     with open(OUT / f"{tag}_optuna_results.json", "w") as f:
#         json.dump(results, f, indent=2)
#     print(f"Saved: {tag}_optuna_results.json\n")
#     return results

# effnet_tuned_results = train_final_and_evaluate(
#     FEAT_DIM_EFF, study_eff.best_params, feats_tr_eff, feats_val_eff, feats_test_eff,
#     "effnet_b0", "efficientnet_b0_optuna_tuned"
# )
# swint_tuned_results = train_final_and_evaluate(
#     FEAT_DIM_SWIN, study_swin.best_params, feats_tr_swin, feats_val_swin, feats_test_swin,
#     "swin_t", "swin_t_optuna_tuned"
# )

In [ ]:
# Cell 14 : Retrain final Stage 2 models with best hyperparameters, evaluate on val only

def train_final_and_evaluate(feat_dim, best_params, feats_tr_, feats_val_,
                              tag, model_name, epochs=30, patience=7):
    attn_dim = best_params["attn_dim"]
    lr = best_params["lr"]
    dropout = best_params["dropout"]
    weight_decay = best_params["weight_decay"]

    model = BagClassifier(feat_dim, attn_dim, dropout).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimiser = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, patience=3)

    train_ds = CachedBagDataset(feats_tr_, y_tr, bag_tr, train_bag_indices)
    val_ds   = CachedBagDataset(feats_val_, y_val, bag_val, val_bag_indices)
    train_dl = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_cached)
    val_dl   = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_cached)

    weights_path = OUT / f"{tag}_optuna_stage2.pth"
    best_val_loss, patience_ctr = float("inf"), 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for h_list, labels in train_dl:
            h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
            optimiser.zero_grad()
            logit, _ = model(h)
            loss = criterion(logit, label)
            loss.backward()
            optimiser.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_ds)

        model.eval()
        val_loss, val_probs, val_true = 0.0, [], []
        with torch.no_grad():
            for h_list, labels in val_dl:
                h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
                logit, _ = model(h)
                val_loss += criterion(logit, label).item()
                val_probs.append(torch.sigmoid(logit).item())
                val_true.append(label.item())
        val_loss /= len(val_ds)
        val_probs, val_true = np.array(val_probs), np.array(val_true)
        val_auc = roc_auc_score(val_true, val_probs)
        scheduler.step(val_loss)

        print(f"[{tag}] Ep {epoch:02d}/{epochs}  train={train_loss:.4f}  val={val_loss:.4f}  auc={val_auc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss, patience_ctr = val_loss, 0
            torch.save(model.state_dict(), weights_path)
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"  Early stopping at epoch {epoch}")
                break

    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.eval()

    val_bag_probs, val_bag_true = [], []
    with torch.no_grad():
        for h_list, labels in val_dl:
            h = h_list[0].to(DEVICE)
            logit, _ = model(h)
            val_bag_probs.append(torch.sigmoid(logit).item())
            val_bag_true.append(labels[0].item())
    val_bag_probs, val_bag_true = np.array(val_bag_probs), np.array(val_bag_true)

    fpr_c, tpr_c, thresholds_c = roc_curve(val_bag_true, val_bag_probs)
    idx = np.argmax(tpr_c >= 0.90)
    optimal_threshold = float(thresholds_c[idx])

    val_metrics_cal = compute_all_metrics(val_bag_true, val_bag_probs, optimal_threshold, f"[{tag}] Val calibrated")

    results = {
        "model": model_name,
        "feat_dim": feat_dim,
        "optuna_best_params": best_params,
        "val_metrics_cal": val_metrics_cal,
        "optimal_threshold": optimal_threshold,
    }
    with open(OUT / f"{tag}_optuna_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"Saved: {tag}_optuna_results.json\n")
    return results


effnet_tuned_results = train_final_and_evaluate(
    FEAT_DIM_EFF, study_eff.best_params, feats_tr_eff, feats_val_eff,
    "effnet_b0", "efficientnet_b0_optuna_tuned"
)
swint_tuned_results = train_final_and_evaluate(
    FEAT_DIM_SWIN, study_swin.best_params, feats_tr_swin, feats_val_swin,
    "swin_t", "swin_t_optuna_tuned"
)

Cell 15 : Baseline vs tuned comparison table

In [ ]:
# baseline_val = {
#     "EfficientNet-B0": {"auc": 0.9996, "f1": 0.9831, "sensitivity": 0.9864, "specificity": 0.9870, "fpr": 0.0130, "ece": 0.0159},
#     "Swin-T":          {"auc": 1.0000, "f1": 0.9800, "sensitivity": 1.0000, "specificity": 0.9740, "fpr": 0.0260, "ece": 0.0066},
# }
# tuned_test = {
#     "EfficientNet-B0": effnet_tuned_results["test_metrics"],
#     "Swin-T": swint_tuned_results["test_metrics"],
# }

# print(f"{'Model':<18}{'Metric':<14}{'Baseline':>10}{'Tuned':>10}{'Delta':>10}")
# print("-" * 62)
# for name in ["EfficientNet-B0", "Swin-T"]:
#     for k in ["auc", "f1", "sensitivity", "specificity", "fpr", "ece"]:
#         b, t = baseline_test[name][k], tuned_test[name][k]
#         print(f"{name:<18}{k:<14}{b:>10.4f}{t:>10.4f}{t-b:>+10.4f}")

# Cell 15 : baseline vs tuned — compare val_metrics_cal, not test_metrics
baseline_val = {
    "EfficientNet-B0": {...},  # NB03 val_metrics_cal values
    "Swin-T": {...},           # NB05 val_metrics_cal values
}
tuned_val = {
    "EfficientNet-B0": effnet_tuned_results["val_metrics_cal"],
    "Swin-T": swint_tuned_results["val_metrics_cal"],
}
# print loop: baseline_val[name][k] vs tuned_val[name][k]

Part C : ConvNeXt-Nano Optuna (conditional, GPU budget permitting)
Cell 16 : Only run if RUN_CONVNEXT_OPTUNA = True

In [ ]:
# Part B — ConvNeXt-Nano Optuna hyperparameter search
# IMPORTANT:
# - Train + validation features only
# - Test features are NOT extracted or used
# - Final test evaluation happens only in NB09

if RUN_CONVNEXT_OPTUNA:

    # ---------------------------------------------------------
    # 1. Create Optuna study
    # ---------------------------------------------------------
    sampler_cnx = TPESampler(seed=SEED)
    pruner_cnx = MedianPruner(n_warmup_steps=5)

    study_cnx = optuna.create_study(
        direction="minimize",
        sampler=sampler_cnx,
        pruner=pruner_cnx
    )

    # ---------------------------------------------------------
    # 2. Create objective using TRAIN + VALIDATION only
    # ---------------------------------------------------------
    objective_cnx = make_objective(
        FEAT_DIM,
        feats_tr_cnx,
        y_tr,
        bag_tr,
        train_bag_indices,
        feats_val_cnx,
        y_val,
        bag_val,
        val_bag_indices
    )

    # ---------------------------------------------------------
    # 3. Run Optuna search
    # ---------------------------------------------------------
    N_TRIALS_CONVNEXT = 25

    print(
        f"Starting ConvNeXt-Nano Optuna search "
        f"({N_TRIALS_CONVNEXT} trials)..."
    )

    t0 = time.time()

    study_cnx.optimize(
        objective_cnx,
        n_trials=N_TRIALS_CONVNEXT
    )

    print(
        f"\nConvNeXt-Nano Optuna search complete — "
        f"{(time.time() - t0) / 60:.1f} min"
    )

    print("\nBest validation objective:", study_cnx.best_value)
    print("Best params:")
    for k, v in study_cnx.best_params.items():
        print(f"  {k}: {v}")

    # ---------------------------------------------------------
    # 4. Train final Stage 2 using best parameters
    #    TRAIN + VALIDATION only
    # ---------------------------------------------------------
    convnext_tuned_results = train_final_and_evaluate(
        FEAT_DIM,
        study_cnx.best_params,
        feats_tr_cnx,
        feats_val_cnx,
        "convnext_nano",
        "convnext_nano_optuna_tuned"
    )

else:
    print(
        "RUN_CONVNEXT_OPTUNA is False — skipping. "
        "ConvNeXt-Nano stops at the manual-fix (v2) result from Part A."
    )